# Pandas DataFrames: A Complete Guide

Where everything from the Series section comes together. A DataFrame is the
equivalent of an Excel or SQL table, used to store and analyse data. We rarely
work with standalone arrays or Series as analysts, but understanding them is
what lets us get the most out of DataFrames.

**Module overview**

1. **DataFrame Basics** - properties, creation, and reading flat files
2. **Exploring DataFrames** - `head`, `tail`, `sample`, `info`, `describe`
3. **Accessing Data** - columns, `.iloc`, and `.loc`
4. **Dropping Data** - rows, columns, and duplicates
5. **Missing Data** - identification and handling across columns
6. **Filtering** - logical masks and the `query()` method
7. **Sorting** - by index and by values, single and multi-column
8. **Modifying Columns** - renaming and reordering

## DataFrame Basics

A DataFrame is the tabular data structure in Pandas, with both columns and
rows. If a Series is a single column of data, a DataFrame is a table of data,
a collection of those Series. Each column is itself a Pandas Series, so
everything learned in the Series section applies once we access a single
column.

**Two axes**

- **axis 0** - the row axis. The default index is a range of integers, but it
  can be customised just like a Series index.
- **axis 1** - the column axis. The column index is made up of the Series
  names, usually an object data type because column names tend to be strings.

Some operations target axis 0 (e.g. a traditional aggregation summing across
rows), others target axis 1 (e.g. summing across columns).

**Key properties**

- `.shape` - number of rows and columns (the index is not counted as a column)
- `.index` - the row index
- `.columns` - the column index (the Series names)
- `.axes` - both the row and column index together
- `.dtypes` - the data type of each column, which can differ column to column

In [ ]:
import numpy as np
import pandas as pd

### Creating a DataFrame from a dictionary

We can build a DataFrame from a Python dictionary using the `DataFrame`
function. The keys become the column headers and the lists stored as values
become the row data. This is rare in practice - usually we read from an
external source - but it is handy for translating a base Python structure into
a DataFrame.

In [ ]:
df = pd.DataFrame(
    {
        "id": [1, 2],
        "store_nbr": [1, 2],
        "family": ["poultry", "produce"],
    }
)
df

### Reading a flat file with `read_csv`

More often we read data from a flat file or a database. The most basic syntax
for `read_csv` is a file path. Pandas will guess the data types when creating
the DataFrame, and it is rarely perfect, so we often convert types later as we
learn more about the data. Note that Pandas comfortably handles millions of
rows, well beyond the roughly one-million-row ceiling of spreadsheet software.

In [ ]:
oil = pd.read_csv("oil.csv")
oil

Confirm the shape, the index, and the column index. `.index` and `.columns`
are two of the more frequently used attributes, and both can be reassigned.

In [ ]:
oil.shape

In [ ]:
oil.index

In [ ]:
oil.columns

`.axes` returns both indices at once, and `.dtypes` shows how each column was
read in. The date column comes in as an object (text); the price comes in as a
float.

In [ ]:
oil.axes

In [ ]:
oil.dtypes

## Challenge: DataFrame Basics

An email from Chandler Capital, our accountant. Subject: Transactions Data.

> Hi there, I heard you did some great work for our finance department. I need
> some help analysing our transactions to make sure there aren't any
> irregularities in the numbers. The data is stored in transactions.csv. Can
> you read this data in and remind me how many rows are in the data, which
> columns are in the data, and their data types? More to come soon, thanks.

### Solution: DataFrame Basics

Read the file in. A common alternative is to store the path in a variable
first, which is handy when looping through multiple files.

In [ ]:
transactions = pd.read_csv("transactions.csv")
transactions

Number of rows and columns.

In [ ]:
transactions.shape

Data types of each column. Store number and transactions read in as integers;
date reads in as an object, which is almost always the case for dates in a
flat file.

In [ ]:
transactions.dtypes

## Exploring DataFrames: Head, Tail, and Sample

After reading in a DataFrame, a few methods let us quickly understand what is
inside.

- `.head(n)` - first `n` rows (5 by default). The preferred quick spot check
  for whether data was read in correctly.
- `.tail(n)` - last `n` rows (5 by default). Useful for time series, to see
  the most recent dates and confirm the end is populated like the start.
- `.sample(n)` - a random sample of rows (1 by default). Built on NumPy's
  random number generators; set `random_state` for reproducibility.

In [ ]:
transactions.head()

Pass an integer to change the number of rows. Expanding to 10 rows here shows
only one row for 1 January but multiple rows on later dates, something worth
exploring further.

In [ ]:
transactions.head(10)

The tail method, here returning the last three rows.

In [ ]:
transactions.tail(3)

A random sample. Setting `random_state` recreates the exact same sample in a
later session.

In [ ]:
transactions.sample(5, random_state=1)

## Exploring DataFrames: Info and Describe

- `.info()` - key DataFrame metadata: the row and column index, each column's
  position, name, non-null count, and data type, a summary count of each data
  type, and memory usage. If a DataFrame has more than 1.7 million rows, Pandas
  suppresses the non-null counts unless we pass `show_counts=True`.
- `.describe()` - descriptive statistics for the numeric columns by default:
  count, mean, standard deviation, min, max, and the 25th/50th/75th
  percentiles. Pass `include="all"` for categorical statistics too.

`.info()` combines what we would get from shape and axes, plus data types and
non-null counts. Comparing the non-null count to the total rows is a quick way
to assess missing data (for the Pandas/NumPy missing types).

In [ ]:
transactions.info()

`.describe()` summarises the actual data. A typical workflow is head first,
then info, then describe. For transactions, the mean exceeding the median
signals a right-skewed distribution driven up by large values.

In [ ]:
transactions.describe()

Use `include="all"` to add statistics for non-numeric columns: unique count,
the most common (top) value, and its frequency. `.round()` suppresses
scientific notation.

In [ ]:
transactions.describe(include="all").round(2)

## Challenge: Exploring a DataFrame

A follow-up from Chandler. He wants the first five rows inspected, a check for
missing values, the number of unique dates, and the key statistics on the
transactions column.

### Solution: Exploring a DataFrame

Inspect the first five rows. Only one store had transactions on 1 January;
all others start on 2 January, and the data is ordered by date then store
number.

In [ ]:
transactions.head()

Check for missing values. `.isna().sum()` returns a count by column.

In [ ]:
transactions.isna().sum()

The info method confirms the same thing plus more - all non-null counts match
the row count, so there is no missing data.

In [ ]:
transactions.info()

Unique dates and the transaction distribution via describe. There are a little
over 1,600 unique dates (roughly four to five years), a minimum of 5
transactions and a maximum over 8,000, with a right-skewed distribution.

In [ ]:
transactions.describe(include="all").round(2)

## Accessing DataFrame Columns

We can access columns with bracket or dot notation.

- **Bracket notation** `df["col"]` - always works, including names with spaces
  or special characters. Recommended as the default.
- **Dot notation** `df.col` - cleaner but the column name must be a valid
  Python variable (no spaces, no special characters, not an existing method).

Accessing a single column returns a Series, so every Series operation applies.
Passing a **list** of column names returns a DataFrame.

In [ ]:
transactions["transactions"]

Dot notation returns the same Series when the name is a valid identifier.

In [ ]:
transactions.transactions

Once we have a single column, Series operations work directly - unique counts,
means, value counts, and sums.

In [ ]:
transactions["store_nbr"].nunique()

In [ ]:
transactions["transactions"].mean()

In [ ]:
transactions["store_nbr"].value_counts().head()

In [ ]:
transactions["transactions"].sum()

Pass a list of column names to select multiple columns. A DataFrame is
returned, with the columns in the order requested.

In [ ]:
transactions[["transactions", "store_nbr"]]

## Accessing Data with `.iloc` and `.loc`

With DataFrames, both accessors take a second parameter for columns. The first
parameter selects rows, the second selects columns.

- **`.iloc[rows, cols]`** - positional, exclusive stop point. Best when we
  know positions, or to grab the first/last N rows of a custom index.
- **`.loc[rows, cols]`** - label based, inclusive stop point. Preferred for
  most work because we usually know column names but not their positions.

A single colon `:` in a parameter means "everything" on that axis.

Positional access: the first ten rows, all columns. Supplying no column
argument also returns all columns, but stating the colon explicitly is a good
habit.

In [ ]:
oil.iloc[:10, :]

Grab the last column with a negative index. Wrapping it differs from a slice -
this returns a Series.

In [ ]:
oil.iloc[:, -1]

Label-based access with `.loc`. Accessing a single column returns a Series;
wrapping the name in a list returns a DataFrame. Remember the stop point is
inclusive.

In [ ]:
oil.loc[:, "oil_price"]

A list of columns returns a DataFrame in the requested order.

In [ ]:
oil.loc[:, ["oil_price", "date"]]

A slice of columns - no brackets - from date through oil_price (inclusive).

In [ ]:
oil.loc[:, "date":"oil_price"]

## Challenge: Accessing DataFrame Data

A new email from Chandler. Subject: A Few More Stats.

> Hi there. I noticed that the first row is the only one from January 1st,
> 2013. Can you give me a copy of the DataFrame that excludes that row and only
> includes store number and transactions? Also, can you report the number of
> unique store numbers? Finally, report the total number of transactions in
> millions. Thanks.

### Solution: Accessing DataFrame Data

Exclude the first row and keep two columns with `.loc`. The label-based slice
starts at row 1 and runs to the end, selecting store_nbr through transactions.

In [ ]:
transactions.loc[1:, "store_nbr":"transactions"]

The same result with `.iloc`. The row start is identical; the column slice is
positional and the stop point is exclusive.

In [ ]:
transactions.iloc[1:, 1:]

Number of unique store numbers.

In [ ]:
transactions.loc[:, "store_nbr"].nunique()

Total transactions in millions.

In [ ]:
transactions.loc[:, "transactions"].sum() / 1_000_000

## Dropping Columns and Rows

The `drop` method removes rows or columns. Specify `axis=0` for rows and
`axis=1` for columns. Dropping is **not** in place by default - it returns a
DataFrame - unless we pass `inplace=True`, in which case nothing is returned.

Dropping unnecessary columns reduces memory and makes a DataFrame easier to
reason about. Ideally, useless columns should not be read in at all. We drop
rows less often, since we usually prefer to filter on logical criteria.

Drop a column, not in place, so the original DataFrame is unchanged.

In [ ]:
oil.drop("oil_price", axis=1).head()

A safer pattern than `inplace=True` is to assign the result to a new
DataFrame, keeping the original intact while developing a workflow.

In [ ]:
oil_dates = oil.drop("oil_price", axis=1)
oil_dates.head()

Drop a single row with `axis=0`. The index now has a gap; `reset_index` can
restore a consecutive index.

In [ ]:
oil.drop(1, axis=0).head()

## Identifying and Dropping Duplicates

- `.duplicated()` - returns True for rows that duplicate an earlier row. The
  first occurrence is not counted as a duplicate. Pass `subset` to look within
  specific columns.
- `.drop_duplicates()` - removes duplicate rows. Useful arguments: `subset`
  (which columns define a duplicate), `keep` (`"first"` default, or `"last"`),
  and `ignore_index=True` (reset the index after dropping).

A quick scan for duplicates compares the number of rows to the number of
unique values in a column that is supposed to be unique.

Build a small product DataFrame where the dairy row repeats.

In [ ]:
product = pd.DataFrame(
    {
        "product": ["dairy", "dairy", "vegetables", "fruits", "dairy"],
        "price": [2.56, 2.56, 1.40, 3.10, 2.99],
    }
)
product

`.duplicated()` flags the second row as an exact duplicate of the first.

In [ ]:
product.duplicated()

With a `subset`, look for repeats within a single column.

In [ ]:
product.duplicated(subset="product")

`drop_duplicates()` with no arguments removes exact duplicate rows.

In [ ]:
product.drop_duplicates()

A more specific call: define duplicates by the product column, keep the last
instance, and reset the index.

In [ ]:
product.drop_duplicates(subset="product", keep="last", ignore_index=True)

## Challenge: Dropping Data

An email from Chandler. Subject: Reducing The Data.

> Hi there. Can you drop the first row of data this time? A slice is fine for
> viewing data, but we want this permanently removed. Also, drop the date
> column, but not in place. Then, can you get me a DataFrame that includes only
> the last row for each of our stores? I want to take a look at the most recent
> data for each. Thanks.

### Solution: Dropping Data

Permanently drop the first row, in place.

In [ ]:
transactions.drop(0, axis=0, inplace=True)
transactions.head()

Drop the date column, not in place, so it remains in the underlying DataFrame.

In [ ]:
transactions.drop("date", axis=1).head()

Confirm date is still present.

In [ ]:
transactions.head()

Keep only the last row for each store by dropping duplicates on store number
and keeping the last instance.

In [ ]:
transactions.drop_duplicates(subset="store_nbr", keep="last")

## Missing Data

Handling missing data across a DataFrame extends the Series approach with a
few more options.

- `.isna().sum()` - count of missing values by column (returns a Series).
- `.fillna(value)` - fills all missing values by default. Pass a **dictionary**
  to fill specific columns with specific values.
- `.dropna()` - drops rows with any missing value by default. Pass `subset` to
  only consider specific columns.

A product DataFrame with missing values in two columns.

In [ ]:
product_na = pd.DataFrame(
    {
        "product_id": [1, 2, 3, 4, 5],
        "product": ["dairy", np.nan, "vegetables", np.nan, "fruits"],
        "price": [2.56, 1.40, np.nan, 3.10, np.nan],
    }
)
product_na

Count missing values by column.

In [ ]:
product_na.isna().sum()

Fill specific columns by passing a dictionary - here filling price with 0
while leaving product untouched.

In [ ]:
product_na.fillna({"price": 0})

`dropna()` drops any row with a missing value, which can be overkill. Use
`subset` to only drop rows missing a particular column.

In [ ]:
product_na.dropna(subset=["price"])

## Challenge: Missing Data

An email from Chandler. Subject: Missing Price Data.

> Hi again, I was reviewing some of Rachel Revenue's work on the oil price data
> as I was closing the books, and I noticed quite a few missing dates and
> values so now I'm concerned. Can you tell if any dates or prices are missing?
> I'd like to see the mean oil price if we fill in the missing values with zero
> and compare it to the mean oil price if we fill them in with the mean.
> Thanks.

### Solution: Missing Data

Re-read the oil data and get the missing-value counts by column.

In [ ]:
oil = pd.read_csv("oil.csv")
oil.isna().sum()

Mean oil price when missing values are filled with zero.

In [ ]:
oil["oil_price"].fillna(0).mean()

Mean oil price when missing values are filled with the column mean. Filling
with zero produces a lower mean, since zero sits below every real price.

In [ ]:
oil["oil_price"].fillna(oil["oil_price"].mean()).mean()

## Filtering DataFrames

Filtering passes a logical test into `.loc`. With a single parameter, `.loc`
filters rows. We can also select columns in the same call, and combine
conditions into a Boolean mask with `&` (and) and `|` (or), each condition
wrapped in parentheses. Logical tests can also compare one column against
another.

Add a benchmark column equal to 100, then filter to prices above it.

In [ ]:
oil["benchmark"] = 100
oil.loc[oil["oil_price"] > 100]

Compare across columns instead of a literal - the same result.

In [ ]:
oil.loc[oil["oil_price"] > oil["benchmark"]]

Filter on the year by slicing the string date, then select rows from 2013.

In [ ]:
oil.loc[oil["date"].str[:4] == "2013"].tail()

Combine conditions with a mask: rows from 2013 where the price exceeded 100.

In [ ]:
mask = (oil["date"].str[:4] == "2013") & (oil["oil_price"] > 100)
oil.loc[mask]

## Pro Tip: The `query()` Method

The `query` method filters with SQL-like syntax, using `and`/`or` keywords and
the `in` operator, without repeating the DataFrame name. It is often more
efficient than a Boolean mask and very readable. Reference external variables
with the `@` symbol. A few functions (like string slicing) are unsupported,
and some teams dislike the pseudo-language, but it is convenient.

Filter to prices above the benchmark, or rows where the year is 2013. String
slicing inside `query` is unsupported, so pull the year into its own column
first and reference it directly.

In [ ]:
oil["year"] = oil["date"].str[:4]
oil.query("oil_price > benchmark or year == '2013'").head()

Reference an external variable with `@`. Here we filter to prices above the
overall average price.

In [ ]:
avg_price = oil["oil_price"].mean()
oil.query("oil_price > @avg_price").head()

## Challenge: Filtering DataFrames

An email from Chandler. Subject: Store 25 Deep Dive.

> I need some quick research on store 25. First, calculate the percentage of
> times all stores had more than 2000 transactions. Then calculate the
> percentage of times store 25 had more than 2000 transactions, and calculate
> the sum of transactions on these days. Finally, sum the transactions for
> stores 25 and 31 that occurred in May or June and had less than 2000
> transactions.

### Solution: Filtering DataFrames

Re-read the data and drop the lone 1 January row to match the running state.

In [ ]:
transactions = pd.read_csv("transactions.csv")
transactions = transactions.drop(0, axis=0).reset_index(drop=True)

Percentage of times all stores had more than 2000 transactions - the mean of
the Boolean condition.

In [ ]:
(transactions["transactions"] > 2000).mean()

Percentage of times store 25 had more than 2000 transactions. The numerator is
the stricter mask; the denominator is just the store-25 rows.

In [ ]:
mask = (transactions["transactions"] > 2000) & (transactions["store_nbr"] == 25)
denom = (transactions["store_nbr"] == 25)
mask.sum() / denom.sum()

Sum of transactions on the days store 25 exceeded 2000.

In [ ]:
transactions.loc[mask, "transactions"].sum()

Sum the transactions for stores 25 and 31 in May or June with fewer than 2000
transactions. The query method handles this complex filter cleanly. String
slicing is unsupported inside `query`, so pull the month into a column first.

In [ ]:
transactions["month"] = transactions["date"].str[5:7]
(
    transactions.query(
        "store_nbr in [25, 31] and month in ['05', '06'] and transactions < 2000"
    )["transactions"].sum()
)

## Sorting DataFrames

- `.sort_index()` - sort by the row index, or by the column index with
  `axis=1`. Ascending by default; pass `ascending=False` to reverse.
- `.sort_values(by=...)` - sort by one or more columns. Pass a list of columns
  to sort by several, and a list to `ascending` to set direction per column.

Sorting is not in place by default. NaNs are pushed to the bottom of a sort.

Sort the oil DataFrame by index in reverse order.

In [ ]:
oil.sort_index(ascending=False).head()

Sort the column index into reverse alphabetical order with `axis=1`.

In [ ]:
oil.sort_index(axis=1, ascending=False).head()

Sort by values in the price column, ascending. NaNs fall to the bottom.

In [ ]:
oil.sort_values("oil_price").head()

Add a month column, then sort by multiple columns: month ascending, then price
descending within each month.

In [ ]:
oil["month"] = pd.to_datetime(oil["date"]).dt.month
oil.sort_values(["month", "oil_price"], ascending=[True, False]).head()

## Challenge: Sorting DataFrames

An email from Chandler. Subject: Sorting help!

> Hi there. Can you get me a data set that includes the five days with the
> highest transaction counts? Are there any similarities between them? Then can
> you get me a data set sorted by date from earliest to most recent, but with
> the highest transactions first and the lowest transactions last for each day?
> Finally, sort the columns in reverse alphabetical order. Thanks.

### Solution: Sorting DataFrames

The five highest transaction counts. Sorting scrambles the index, so `.iloc`
is the right tool to grab the top five rows. They cluster on 23 December (the
pre-Christmas rush) at a couple of busy stores.

In [ ]:
transactions.sort_values("transactions", ascending=False).iloc[:5]

Sort by date ascending, then transactions descending within each date.

In [ ]:
transactions.sort_values(["date", "transactions"], ascending=[True, False]).head()

Sort the columns into reverse alphabetical order.

In [ ]:
transactions.sort_index(axis=1, ascending=False).head()

## Renaming and Reordering Columns

- **Rename by assignment** - assign a list to `.columns` (in place, must match
  the column order).
- **`.rename(columns=...)`** - pass a dictionary mapping old names to new
  names. Not in place by default, which makes it convenient for chaining.
- **`.reindex(labels=..., axis=1)`** - reorder columns by passing the names in
  the desired order.

A list comprehension on `.columns` is one way to apply uniform formatting,
such as upper-casing every name.

In [ ]:
product.columns = [col.upper() for col in product.columns]
product.head()

The `rename` method maps old names to new names via a dictionary, not in place.

In [ ]:
product.rename(columns={"PRODUCT": "product_name", "PRICE": "cost"})

## Challenge: Renaming and Reordering Columns

An email from Chandler. Subject: Cosmetic Changes.

> Hi again. Just some quick work, but can you send me the transaction data with
> the columns renamed? Rename transactions to transaction_count and store_nbr
> to store_number. Also, could you reorder the columns so date is first, then
> store number, then transaction count? Thanks.

### Solution: Renaming and Reordering Columns

Chain rename into reindex in a single step, wrapping the chain in parentheses
for clean multi-line formatting.

In [ ]:
(
    transactions.rename(
        columns={"transactions": "transaction_count", "store_nbr": "store_number"}
    ).reindex(labels=["date", "store_number", "transaction_count"], axis=1)
).head()

## What Comes Next

With DataFrame basics, exploration, access, dropping, missing data, filtering,
sorting, and column modification covered, the next lessons look at creating new
columns, applying custom functions, Pandas data types, and memory optimisation.